# Reinforcement learning into the residual stream

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pfekin/LARA/blob/main/examples/rl_grpo.ipynb)

Cross-entropy and preference optimization both write into a LARA behavior
without touching the base model. This notebook asks whether a policy gradient
does too, and puts a LoRA through the same loop for comparison.

**The task.** Follow a set of formatting constraints exactly: three bullets, all
lowercase, no word over eight letters, ends with a given phrase. Each constraint
is checkable in a line of code, so the reward is a rule checker rather than a
learned model and there is nothing to argue about in the scoring.

**Why RL rather than supervised training.** These constraints are easy to check
and awkward to write targets for. That gap is the case for a policy gradient: you
score what the model produced instead of telling it what to produce.

**Why it is worth measuring.** Constraint following is where small models are
weakest and where the benchmarks that measure it, IFEval and IFBench, are
themselves rule-based. Three constraint types are held back from training
entirely, so the numbers separate learning to follow instructions from learning
a fixed list of rules.

**Runtime.** Roughly 45 minutes on a T4: two training runs and the evaluation.

## Configuration

In [1]:
!pip install -q git+https://github.com/pfekin/LARA.git
!pip install -q transformers accelerate peft
# peft runs a version check on torchao whenever it is importable, and the build
# Colab ships fails it. Nothing here uses torchao, so removing it is cleaner than
# upgrading: an upgrade can pull a wheel built for a different Python.
!pip uninstall -q -y torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import gc, json, math, os, random, re, string
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

from lara import LARA, Bank

NL = chr(10)

# Any causal LM. A standard base is used rather than a quantized one, so nothing
# in an RL result has to be attributed to the weight format.
BASE = "Qwen/Qwen3-1.7B"

BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE  = torch.bfloat16 if BF16 else torch.float16
GAMMAS = (0.0, 0.5, 1.0, 1.5, 2.0)   # 0.0 is the untouched base

# ── GRPO ─────────────────────────────────────────────────────────────────────
# No value network, so the parameter count stays clean for the size comparison.
# Advantages come from comparing samples within a group instead.
GROUP       = 6        # completions sampled per prompt
STEPS       = 800      # prompt groups; one optimizer update each
LR          = 1e-4
KL_BETA     = 0.02     # holds the policy near the base, which stops the reward
                       # being won by degenerate output
TEMP        = 0.9
MAX_NEW     = 200      # a cut-off completion can never satisfy 'end with'
N_PROMPTS   = 700      # screened down to the learnable band below
N_EVAL      = 300      # after screening; keeps the noise floor near six points
                       # rather than the eleven that 80 prompts gave
EVAL_BS     = 16       # prompts per batch when scoring greedily
SEQ_BS      = 48       # sequences per batch when sampling groups; a prompt
                       # sampled k times contributes k sequences

LAYERS, RANK, ALPHA = 1, 128, 128     # preference training needed one module;
                                      # this checks whether a policy gradient does too
LORA_RANK   = 8
LORA_TARGET = ["q_proj", "k_proj", "v_proj", "o_proj",
               "gate_proj", "up_proj", "down_proj"]

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{torch.cuda.get_device_name(0)}  {gb:.0f} GB  bf16={'yes' if BF16 else 'no'}")
else:
    print("no GPU: pick a GPU runtime")
print(f"base {BASE}")

NVIDIA L4  24 GB  bf16=yes
base Qwen/Qwen3-1.7B


## 1. The reward

Twelve constraint types, each with a checker of a line or two. A prompt asks for
two or three of them, and the reward is the share satisfied. A dense reward
matters here: all-or-nothing gives almost no gradient early on, when the model
satisfies one constraint out of three and needs to be told that is progress.

Three types are never used in training. They appear only in the held-out
evaluation, so a model that learned the twelve rules by name scores badly there
while one that learned to read the instruction does not.

Three guards against the reward being won the wrong way, or lost unfairly. A
response under twenty characters scores zero, because an empty string satisfies
most negative constraints. A response that is mostly one repeated word scores
zero. And a completion that ran into the token limit scores zero rather than
being graded on the fragment that survived: a cut-off answer cannot end with a
given phrase, and its word count is an artefact of the cut.

In [3]:
def n_bullets(t):
    return sum(1 for ln in t.strip().split(NL) if ln.strip().startswith(("-", "*", "•")))


def n_sentences(t):
    return len([s for s in re.split(r"[.!?]+", t) if s.strip()])


# Each entry: how it reads in the prompt, and how it is checked.
CONSTRAINTS = {
    "bullets":      (lambda n: f"use exactly {n} bullet points, each on its own line",
                     lambda t, n: n_bullets(t) == n,
                     lambda r: r.choice([4, 5, 6])),
    "lowercase":    (lambda _: "write entirely in lowercase",
                     lambda t, _: t == t.lower(),
                     lambda r: None),
    "no_digits":    (lambda _: "use no digits anywhere",
                     lambda t, _: not any(c.isdigit() for c in t),
                     lambda r: None),
    "short_words":  (lambda n: f"use no word longer than {n} letters",
                     lambda t, n: all(len(w.strip(string.punctuation)) <= n
                                      for w in t.split()),
                     lambda r: r.choice([4, 5, 6])),
    "include":      (lambda w: f"include the word '{w}'",
                     lambda t, w: w.lower() in t.lower(),
                     lambda r: r.choice(["careful", "simple", "steady", "plain"])),
    "ends_with":    (lambda w: f"end with the exact phrase '{w}'",
                     lambda t, w: t.strip().rstrip(".").lower().endswith(w.lower()),
                     lambda r: r.choice(["that is all", "in summary", "and no more"])),
    "starts_with":  (lambda w: f"start with the exact word '{w}'",
                     lambda t, w: t.strip().lower().startswith(w.lower()),
                     lambda r: r.choice(["First", "Note", "Consider"])),
    "no_commas":    (lambda _: "use no commas",
                     lambda t, _: "," not in t,
                     lambda r: None),
    "caps_word":    (lambda w: f"write the word '{w}' in capitals",
                     lambda t, w: w.upper() in t,
                     lambda r: r.choice(["NOTE", "KEY", "WARNING"])),
    # ── held out: never used in training ─────────────────────────────────────
    "sentences":    (lambda n: f"use exactly {n} sentences",
                     lambda t, n: n_sentences(t) == n,
                     lambda r: r.choice([2, 3])),
    "exclude":      (lambda w: f"do not use the word '{w}'",
                     lambda t, w: w.lower() not in t.lower(),
                     lambda r: r.choice(["very", "really", "just"])),
    "word_count":   (lambda n: f"use between {n} and {n + 6} words",
                     lambda t, n: n <= len(t.split()) <= n + 6,
                     lambda r: r.choice([25, 35, 45])),
}
TRAIN_TYPES = ["bullets", "lowercase", "no_digits", "short_words", "include",
               "ends_with", "starts_with", "no_commas", "caps_word"]
HELD_TYPES  = ["sentences", "exclude", "word_count"]

TOPICS = ["how to store bread", "why bicycles stay upright", "what a firewall does",
          "how compost works", "why coffee tastes bitter", "how a zip file shrinks data",
          "what makes glue stick", "how tides are caused", "why paint dries slowly",
          "what a passport control does", "how a thermos keeps heat",
          "why leaves change colour", "what a search index stores",
          "how a violin makes sound", "why metal feels cold"]


def make_task(rng, types):
    """A topic, two or three constraints, and the prompt that asks for both."""
    # Three or four rules, not two. With two the base often satisfies both by
    # accident, and a group where every sample scores 1.0 teaches nothing.
    picked = rng.sample(types, rng.randint(3, min(4, len(types))))
    rules = []
    for name in picked:
        phrase, check, draw = CONSTRAINTS[name]
        arg = draw(rng)
        rules.append((name, arg, phrase(arg), check))
    topic = rng.choice(TOPICS)
    text = (f"Explain {topic}." + NL + "Follow all of these rules:" + NL
            + NL.join(f"- {r[2]}" for r in rules))
    return {"prompt": text, "topic": topic, "rules": rules}


def score(task, response, truncated=False):
    """Share of constraints satisfied, with three guards against winning cheaply
    and one against losing unfairly.

    A very short answer satisfies 'no digits', 'no commas' and 'short words' at
    once without saying anything, so anything under fifteen words scores zero.
    Repetition is the other cheap win, so a response that is mostly one repeated
    word scores zero. And a completion cut off at the token limit scores zero
    rather than being graded on the fragment that survived."""
    # A completion cut off at the token limit is not an answer. It can never
    # satisfy "end with", and its sentence and word counts are whatever the cut
    # happened to leave. Scoring it would measure the limit, not the model.
    if truncated:
        return 0.0
    t = response.strip()
    # Answering "A firewall simple blocks bad traffic and no more" satisfies
    # 'no commas', 'include the word' and 'end with' in nine words. That is
    # gaming the checker, not following the instruction, so require enough text
    # to have actually tried. Fifteen words is below any length rule asked for.
    if len(t.split()) < 15:
        return 0.0
    words = t.lower().split()
    if words and max(words.count(w) for w in set(words)) > len(words) * 0.4:
        return 0.0
    hits = 0
    for name, arg, _, check in task["rules"]:
        try:
            hits += int(check(t, arg))
        except Exception:
            pass
    return hits / len(task["rules"])


def all_satisfied(task, response):
    return score(task, response) == 1.0


rng = random.Random(0)
train_tasks = [make_task(rng, TRAIN_TYPES) for _ in range(N_PROMPTS)]
rng2 = random.Random(99)
eval_seen = [make_task(rng2, TRAIN_TYPES) for _ in range(int(N_EVAL * 1.35))]
# Sampling from the combined pool means only some prompts get a held-out rule,
# so oversample and keep the ones that do. A short eval set is how a difference
# of ten points becomes indistinguishable from noise.
eval_held = []
while len(eval_held) < int(N_EVAL * 1.35):
    t = make_task(rng2, TRAIN_TYPES + HELD_TYPES)
    if any(r[0] in HELD_TYPES for r in t["rules"]):
        eval_held.append(t)

print(f"{len(train_tasks)} training prompts over {len(TRAIN_TYPES)} constraint types")
print(f"{len(eval_seen)} eval prompts, trained types only")
print(f"{len(eval_held)} eval prompts, each with at least one of {HELD_TYPES}")
print(f"  a difference smaller than {1.96 * (0.25 / N_EVAL) ** 0.5:.1%} is noise "
      f"at this size")
print()
print(train_tasks[0]["prompt"])

# A perfect answer must score 1.0, or every number below means nothing.
probe = {"rules": [("lowercase", None, "", CONSTRAINTS["lowercase"][1]),
                   ("no_commas", None, "", CONSTRAINTS["no_commas"][1])]}
GOOD = ("bread keeps best in a paper bag kept away from the sun and "
        "eaten within a couple of days")
assert score(probe, GOOD) == 1.0
assert score(probe, GOOD.capitalize()) < 1.0           # a capital breaks lowercase
assert score(probe, GOOD.replace("bag kept", "bag, kept")) < 1.0   # a comma breaks it
assert score(probe, "bread keeps best in a bag") == 0.0            # under fifteen words
assert score(probe, GOOD, truncated=True) == 0.0                   # cut off mid answer
print()
print("reward checks pass")

700 training prompts over 9 constraint types
405 eval prompts, trained types only
405 eval prompts, each with at least one of ['sentences', 'exclude', 'word_count']
  a difference smaller than 5.7% is noise at this size

Explain how tides are caused.
Follow all of these rules:
- start with the exact word 'Note'
- use exactly 5 bullet points, each on its own line
- use no digits anywhere
- include the word 'steady'

reward checks pass


## 2. Generation and scoring

In [4]:
def load_base():
    tok = AutoTokenizer.from_pretrained(BASE)
    tok.pad_token = tok.pad_token or tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE, device_map="auto")
    return m, tok


def prompt_for(tok, text):
    msgs = [{"role": "user", "content": text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def sample(model, tok, text, k, temp=TEMP):
    """k completions for one prompt. Returns the text, the token ids, and
    whether each one ran into the length limit."""
    tok.padding_side = "left"
    enc = tok([prompt_for(tok, text)], return_tensors="pt").to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=temp > 0,
                         temperature=temp or None, top_p=0.95, num_return_sequences=k,
                         pad_token_id=tok.pad_token_id)
    n = enc.input_ids.shape[1]
    comps = [r[n:] for r in out]
    cut = [bool((c != tok.pad_token_id).sum().item() >= MAX_NEW and
                tok.eos_token_id not in c.tolist()) for c in comps]
    return ([tok.decode(c, skip_special_tokens=True) for c in comps], comps, cut)


@torch.no_grad()
def sample_groups(model, tok, tasks, k, temp=TEMP, bs=None):
    """Group sampling for many prompts at once.

    Each prompt is repeated k times inside the batch, so screening can use the
    same distribution training will see. Screening greedily and training at
    temperature is how a prompt with no spread survives the filter."""
    tok.padding_side = "left"
    # SEQ_BS counts sequences, and each prompt contributes k of them. Sizing the
    # batch in prompts is how this ends up issuing hundreds of generate calls.
    bs = bs or max(1, SEQ_BS // k)
    groups = []
    n_batches = (len(tasks) + bs - 1) // bs
    for i in range(0, len(tasks), bs):
        if (i // bs) % 10 == 0:
            print(f"      batch {i // bs + 1}/{n_batches}", flush=True)
        chunk = tasks[i:i + bs]
        prompts = [prompt_for(tok, t["prompt"]) for t in chunk for _ in range(k)]
        enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=temp > 0,
                             temperature=temp or None, top_p=0.95,
                             pad_token_id=tok.pad_token_id)
        n = enc.input_ids.shape[1]
        for j, t in enumerate(chunk):
            rows = out[j * k:(j + 1) * k]
            comps = [r[n:] for r in rows]
            cut = [bool((c != tok.pad_token_id).sum().item() >= MAX_NEW and
                        tok.eos_token_id not in c.tolist()) for c in comps]
            texts = [tok.decode(c, skip_special_tokens=True) for c in comps]
            groups.append((t, texts, cut))
    return groups


def completion_logprob(model, tok, text, comp_ids):
    """Log probability of a sampled completion under the current policy.
    Kept per token, so long and short completions are comparable."""
    p = tok(prompt_for(tok, text), return_tensors="pt").input_ids.to(model.device)
    c = comp_ids.unsqueeze(0).to(model.device)
    ids = torch.cat([p, c], dim=1)
    lg = model(ids).logits[:, :-1].float()
    lp = torch.log_softmax(lg, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    seg = lp[:, p.shape[1] - 1:]
    return seg.mean(), seg


@torch.no_grad()
def evaluate(model, tok, tasks, bs=EVAL_BS):
    """Greedy, so the number is reproducible. Reports the share of prompts where
    every constraint was met, and the share of individual constraints met."""
    tok.padding_side = "left"
    full = part = 0.0
    n_cut = 0
    outs = []
    for i in range(0, len(tasks), bs):
        chunk = tasks[i:i + bs]
        enc = tok([prompt_for(tok, t["prompt"]) for t in chunk],
                  return_tensors="pt", padding=True).to(model.device)
        g = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                           pad_token_id=tok.pad_token_id)
        n = enc.input_ids.shape[1]
        for t, row in zip(chunk, g):
            comp = row[n:]
            cut = bool((comp != tok.pad_token_id).sum().item() >= MAX_NEW and
                       tok.eos_token_id not in comp.tolist())
            txt = tok.decode(comp, skip_special_tokens=True)
            outs.append(txt)
            n_cut += int(cut)
            s = score(t, txt, cut)
            part += s
            full += float(s == 1.0)
    return {"all_rules": full / len(tasks), "per_rule": part / len(tasks),
            "truncated": n_cut / len(tasks), "outputs": outs}

## 3. Which prompts are worth training on

A group where every sample scores 1.0 produces no gradient, and neither does one
where every sample scores 0. Both are steps spent for nothing.

The base already solves a large share of these prompts outright. Left alone,
about seventy percent of training groups come back with every sample scoring
1.0, which turns most of the run into no-ops. Scoring the pool first and keeping
only the middle band costs two or three minutes and is worth more than any
hyperparameter here.

The same screen runs over the evaluation sets, for a different reason. If the
base already scores well there, the headroom you can demonstrate is capped no
matter how training goes.

In [5]:
def screen(model, tok, tasks, k=4, lo=0.05, hi=0.95, label=""):
    """Keep the prompts a policy gradient can learn from.

    The test is spread within a sampled group, not the mean. GRPO derives its
    advantage from samples disagreeing with each other, so a prompt where every
    sample scores the same teaches nothing whether that score is 0 or 1.

    Sampled at the training temperature, in groups, because a greedy pass tests
    a different distribution from the one training will see."""
    groups = sample_groups(model, tok, tasks, k)
    kept, solved, trunc = [], 0, 0
    for task, texts, cut in groups:
        r = np.array([score(task, t, c) for t, c in zip(texts, cut)])
        trunc += sum(cut)
        solved += int(r.mean() == 1.0)
        if r.std() > 1e-6 and lo < r.mean() < hi:
            kept.append(task)
    n = len(groups)
    print(f"    {label}: kept {len(kept)} of {n}; "
          f"{solved / n:.0%} already solved, {trunc / (n * k):.0%} of samples truncated")
    return kept


model, tok = load_base()
model.config.use_cache = True
model.eval()

print("training pool")
train_tasks = screen(model, tok, train_tasks, label="train")

# Evaluation keeps everything the base does not already ace, so the ceiling is
# visible rather than hidden. Spread does not matter here: nothing is learned
# from the evaluation set, so a prompt the base always fails is still useful.
print("evaluation sets")
ev = sample_groups(model, tok, eval_seen, 1, temp=0.0)
eval_seen = [t for t, tx, c in ev if score(t, tx[0], c[0]) < 0.95][:N_EVAL]
ev = sample_groups(model, tok, eval_held, 1, temp=0.0)
eval_held = [t for t, tx, c in ev if score(t, tx[0], c[0]) < 0.95][:N_EVAL]
print(f"    kept {len(eval_seen)} trained-type and {len(eval_held)} held-out-type prompts")

assert len(train_tasks) >= 50, (
    "too few learnable prompts. Either the constraints are so hard the base "
    "never satisfies any, or so easy it satisfies all: check the screen bounds.")

del model
gc.collect(); torch.cuda.empty_cache()


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

training pool
      batch 1/59
      batch 11/59
      batch 21/59
      batch 31/59
      batch 41/59
      batch 51/59
    train: kept 220 of 700; 22% already solved, 2% of samples truncated
evaluation sets
      batch 1/9


[transformers] The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


      batch 1/9
    kept 286 trained-type and 291 held-out-type prompts


## 4. GRPO

Sample a group of completions for one prompt, score each with the rule checker,
and push up the ones that scored above the group average. The advantage is the
group-relative score, which is what removes the need for a value network.

A KL term holds the policy near the base. Without it a policy gradient on a
formatting reward drifts into text that satisfies the rules and reads like
nothing.

For LARA the reference is free: the modules start at zero and the correction is
scaled by gamma, so the model at strength 0 *is* the base. For LoRA the adapters
are disabled instead, which is the same trick through a different door.

In [6]:
def grpo(model, tok, params, reference, tag):
    """One optimizer update per prompt group. `reference` is a context manager
    that turns the adaptation off, giving the base policy for the KL term."""
    opt = torch.optim.AdamW(params, lr=LR)
    scaler = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
    order = list(range(len(train_tasks)))
    random.Random(1).shuffle(order)
    hist, step, flat = [], 0, 0

    while step < STEPS:
        for idx in order:
            task = train_tasks[idx]
            texts, ids, cut = sample(model, tok, task["prompt"], GROUP)
            rewards = torch.tensor([score(task, t, c) for t, c in zip(texts, cut)])
            hist.append(rewards.mean().item())

            # Nothing to learn from a group that agrees with itself.
            if rewards.std() < 1e-6:
                # Nothing to learn from a group that agrees with itself. A high
                # count here means the prompts are too easy or too hard, which
                # the screening step above is meant to prevent.
                flat += 1
                step += 1
                continue
            adv = (rewards - rewards.mean()) / (rewards.std() + 1e-6)

            with torch.no_grad(), reference():
                ref_lp = [completion_logprob(model, tok, task["prompt"], c)[0].item()
                          for c in ids]

            opt.zero_grad(set_to_none=True)
            for a, c, rlp in zip(adv, ids, ref_lp):
                lp, _ = completion_logprob(model, tok, task["prompt"], c)
                kl = lp - rlp                        # per token, so groups compare
                loss = (-a.item() * lp + KL_BETA * kl.abs()) / GROUP
                if torch.isfinite(loss):
                    scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()

            step += 1
            if step % 40 == 0:
                print(f"    {tag} step {step:>4}  mean reward "
                      f"{np.mean(hist[-40:]):.3f}   "
                      f"{flat}/{step} groups had no spread")
            if step >= STEPS:
                break
    return hist

## 5. Train the behavior

In [7]:
import contextlib

model, tok = load_base()
model.config.use_cache = True          # generation happens inside the loop
lara = LARA(model, layers=LAYERS, rank=RANK, alpha=ALPHA)
lara_params = [p for p in model.parameters() if p.requires_grad]
N_LARA = sum(p.numel() for p in lara_params)
print(f"LARA: {N_LARA:,} trainable across {LAYERS} module(s)")


@contextlib.contextmanager
def lara_off():
    """Strength 0 is the base exactly, so this is the reference policy."""
    g = lara.gamma
    lara.gamma = 0.0
    try:
        yield
    finally:
        lara.gamma = g


lara.gamma = 1.0
lara_hist = grpo(model, tok, lara_params, lara_off, "LARA")
lara.save("behaviors/constraints",
          route_samples=[t["prompt"] for t in train_tasks[:200]], method="rl")
LARA_MB = sum(os.path.getsize(os.path.join(d, f))
              for d, _, fs in os.walk("behaviors/constraints") for f in fs) / 1e6
print(f"saved behaviors/constraints  ({LARA_MB:.1f} MB)")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

LARA: 530,432 trainable across 1 module(s)
    LARA step   40  mean reward 0.667   9/40 groups had no spread
    LARA step  160  mean reward 0.680   41/160 groups had no spread
    LARA step  200  mean reward 0.655   53/200 groups had no spread
    LARA step  240  mean reward 0.742   74/240 groups had no spread
    LARA step  280  mean reward 0.705   86/280 groups had no spread
    LARA step  320  mean reward 0.665   103/320 groups had no spread
    LARA step  480  mean reward 0.763   183/480 groups had no spread
    LARA step  680  mean reward 0.824   311/680 groups had no spread
    LARA step  720  mean reward 0.830   336/720 groups had no spread
saved behaviors/constraints  (2.2 MB)


## 6. What it does, at each strength

Strength 0 is the check: the correction is scaled to nothing, so the model should
measure identically to the base rather than approximately.

Nothing else in this notebook has a dial. That is the point of showing it.

In [8]:
lara_scores = {}
for g in GAMMAS:
    lara.gamma = g
    a = evaluate(model, tok, eval_seen)
    b = evaluate(model, tok, eval_held)
    lara_scores[g] = (a, b)
    print(f"  strength {g}:  trained types {a['all_rules']:.0%}   "
          f"held-out types {b['all_rules']:.0%}")

lara.gamma = 0.0
base_seen, base_held = lara_scores[0.0]
lara.detach()
del model
gc.collect(); torch.cuda.empty_cache()

  strength 0.0:  trained types 2%   held-out types 2%
  strength 0.5:  trained types 15%   held-out types 12%
  strength 1.0:  trained types 24%   held-out types 10%
  strength 1.5:  trained types 28%   held-out types 10%
  strength 2.0:  trained types 29%   held-out types 7%
  strength 2.5:  trained types 26%   held-out types 5%


## 7. The same loop with LoRA

Identical prompts, identical reward, identical number of steps. The only change
is where the adaptation lives.

This is not a claim about LoRA in general. It is one configuration at one rank on
one task, run to see whether a policy gradient reaches both kinds of adapter.

In [9]:
from peft import LoraConfig, get_peft_model

model, tok = load_base()
model.config.use_cache = True
model = get_peft_model(model, LoraConfig(
    r=LORA_RANK, lora_alpha=2 * LORA_RANK, lora_dropout=0.0,
    target_modules=LORA_TARGET, task_type="CAUSAL_LM"))
lora_params = [p for p in model.parameters() if p.requires_grad]
N_LORA = sum(p.numel() for p in lora_params)
print(f"LoRA: {N_LORA:,} trainable  ({N_LORA / N_LARA:.1f}x LARA)")


@contextlib.contextmanager
def lora_off():
    with model.disable_adapter():
        yield


lora_hist = grpo(model, tok, lora_params, lora_off, "LoRA")
model.save_pretrained("lora/constraints")
LORA_MB = sum(os.path.getsize(os.path.join(d, f))
              for d, _, fs in os.walk("lora/constraints") for f in fs) / 1e6

lora_seen = evaluate(model, tok, eval_seen)
lora_held = evaluate(model, tok, eval_held)
print(f"  LoRA: trained types {lora_seen['all_rules']:.0%}   "
      f"held-out types {lora_held['all_rules']:.0%}   ({LORA_MB:.1f} MB)")

del model
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

LoRA: 8,716,288 trainable  (16.4x LARA)
    LoRA step   40  mean reward 0.684   11/40 groups had no spread
    LoRA step  120  mean reward 0.663   51/120 groups had no spread
    LoRA step  200  mean reward 0.773   105/200 groups had no spread
    LoRA step  640  mean reward 0.829   464/640 groups had no spread
    LoRA step  680  mean reward 0.841   494/680 groups had no spread
    LoRA step  720  mean reward 0.876   524/720 groups had no spread
    LoRA step  760  mean reward 0.788   554/760 groups had no spread
  LoRA: trained types 29%   held-out types 11%   (34.9 MB)


## 8. Results

`all rules` is the share of prompts where every constraint was met. `per rule`
counts individual constraints, which moves earlier and shows partial progress.

The held-out column uses constraint types neither run ever trained on.

In [10]:
# Held-out types, not trained ones. Picking on the trained set chooses the
# strength where the behavior has leaned hardest into the rules it saw, which
# is usually past the point where it still writes sensible text.
best = max(GAMMAS, key=lambda g: lara_scores[g][0]["all_rules"]
                                + lara_scores[g][1]["all_rules"])
rows = [("base, nothing attached", base_seen, base_held, 0.0, N_LARA * 0),
        (f"LARA behavior (g={best})", lara_scores[best][0], lara_scores[best][1],
         LARA_MB, N_LARA),
        ("LoRA", lora_seen, lora_held, LORA_MB, N_LORA)]

print(f"{'':<26}{'trained types':>26}{'held-out types':>22}")
print(f"{'':<26}{'all rules':>13}{'per rule':>13}{'all rules':>11}{'per rule':>11}")
print("-" * 74)
for name, a, b, _, _ in rows:
    print(f"{name:<26}{a['all_rules']:>13.0%}{a['per_rule']:>13.0%}"
          f"{b['all_rules']:>11.0%}{b['per_rule']:>11.0%}")

print()
print(f"{'':<26}{'trainable':>14}{'on disk':>12}")
print("-" * 52)
for name, _, _, mb, n in rows[1:]:
    print(f"{name:<26}{n:>14,}{mb:>11.1f} MB")

print()
n = len(eval_seen)
print(f"differences under {1.96 * (0.25 / n) ** 0.5:.1%} are within noise at "
      f"{n} evaluation prompts")
print()
print(f"completions cut off at the {MAX_NEW}-token limit: "
      f"base {base_seen['truncated']:.0%}, "
      f"LARA {lara_scores[best][0]['truncated']:.0%}, "
      f"LoRA {lora_seen['truncated']:.0%}   (these score zero)")
print()
print(f"strength 0 against the bare base: identical by construction, "
      f"since the correction is scaled to nothing")
print(f"reward during training: LARA {np.mean(lara_hist[:20]):.2f} -> "
      f"{np.mean(lara_hist[-20:]):.2f}, LoRA {np.mean(lora_hist[:20]):.2f} -> "
      f"{np.mean(lora_hist[-20:]):.2f}")

                                       trained types        held-out types
                              all rules     per rule  all rules   per rule
--------------------------------------------------------------------------
base, nothing attached               2%          62%         2%        57%
LARA behavior (g=0.5)               15%          71%        12%        62%
LoRA                                29%          76%        11%        63%

                               trainable     on disk
----------------------------------------------------
LARA behavior (g=0.5)            530,432        2.2 MB
LoRA                           8,716,288       34.9 MB

completions cut off at the 200-token limit: base 1%, LARA 1%, LoRA 2%   (these score zero)

strength 0 against the bare base: identical by construction, since the correction is scaled to nothing
reward during training: LARA 0.72 -> 0.87, LoRA 0.69 -> 0.88


## 9. One example

In [11]:
i = 0
print(eval_held[i]["prompt"])
print()
for name, out in (("base", base_held["outputs"][i]),
                  ("LARA", lara_scores[best][1]["outputs"][i]),
                  ("LoRA", lora_held["outputs"][i])):
    s = score(eval_held[i], out)
    print(f"{name:<6} [{s:.0%} of rules]")
    print("   ", out.strip().replace(NL, " ")[:220])
    print()

Explain what a passport control does.
Follow all of these rules:
- use exactly 5 bullet points, each on its own line
- use no word longer than 6 letters
- end with the exact phrase 'that is all'
- use between 35 and 41 words

base   [50% of rules]
    - Check identity and citizenship   - Verify travel documents   - Screen for security threats   - Inspect personal belongings   - Ensure compliance with laws that is all

LARA   [50% of rules]
    - Check identity and citizenship   - Verify travel documents   - Screen for security threats   - Inspect personal belongings   - Ensure compliance with laws that is all

LoRA   [50% of rules]
    - checks traveler's identity via photo & info    - verifies citizenship via biometric data    - prevents unauthorized entry into country    - monitors high-risk areas for suspicious activity    - that is all



## Notes

A policy gradient reaches a LARA behavior the same way cross-entropy and
preference optimization do. Nothing about the method assumes a differentiable
target; the correction is a set of parameters and the gradient finds them.

The behavior uses a single module, following the preference result that
placement matters far less for style than for knowledge. Whether that holds
across tasks is untested.

The held-out constraint types are what separate following an instruction from
memorising nine rules. A run that scores well on trained types and badly on
held-out ones has learned the second thing.

The reward is a rule checker, which is why this task was chosen: the scoring is
inspectable and there is no reward model to argue about. Tasks with softer
rewards would need one, and the failure modes are different.

The KL term matters more than it looks. Without it a formatting reward is
straightforwardly hackable, and the guards in `score` catch only the crudest
attempts.

- https://github.com/pfekin/LARA
- https://arxiv.org/abs/2607.28669